In [15]:
import json
import numpy as np
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
from scipy.optimize import least_squares
from scipy.special import erf
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_scales

In [16]:
def generate_trail(shape, flux, x0, y0, length, theta, sigma, background=0.0):
    ny, nx = shape
    y, x = np.mgrid[0:ny, 0:nx]
    xp = (x - x0) * np.cos(theta) + (y - y0) * np.sin(theta)
    yp = -(x - x0) * np.sin(theta) + (y - y0) * np.cos(theta)
    prefactor = flux / (length * 2.0 * sigma * np.sqrt(2.0 * np.pi))
    cross_trail = np.exp(-(yp ** 2) / (2.0 * sigma ** 2))
    u1 = (xp + length / 2.0) / (sigma * np.sqrt(2.0))
    u2 = (xp - length / 2.0) / (sigma * np.sqrt(2.0))
    along_trail = erf(u1) - erf(u2)
    return background + prefactor * cross_trail * along_trail


def fit_trail(image, p0, bounds=None):
    ny, nx = image.shape

    def residuals(params):
        flux, x0, y0, length, theta, sigma, background = params
        if length <= 0 or sigma <= 0 or flux < 0:
            return np.full(image.size, 1e30)
        model = generate_trail(image.shape, flux, x0, y0, length, theta, sigma, background)
        return (model - image).ravel()

    if bounds is None:
        lower = [0.0, 0.0, 0.0, 1e-6, -np.pi, 1e-3, -np.inf]
        upper = [np.inf, nx, ny, max(nx, ny), np.pi, np.inf, np.inf]
        bounds = (lower, upper)

    result = least_squares(residuals, p0, bounds=bounds)

    best_fit = {"flux": result.x[0], "x0": result.x[1], "y0": result.x[2], "length": result.x[3], "theta": result.x[4], "sigma": result.x[5], "background": result.x[6]}

    model = generate_trail(image.shape, best_fit["flux"], best_fit["x0"], best_fit["y0"], best_fit["length"], best_fit["theta"], best_fit["sigma"], best_fit["background"])

    resid = residuals(result.x)
    ndata = resid.size
    npar = result.x.size
    dof = ndata - npar

    if dof > 0:
        chi2 = np.sum(resid**2)
        s_sq = chi2 / dof
        JTJ = result.jac.T @ result.jac
        cov = s_sq * np.linalg.pinv(JTJ)
    else:
        cov = np.full((npar, npar), np.nan)

    return result, best_fit, model, cov


In [17]:
with open("../Step_2_Integration/observability_bright.json", "r") as f:
    data = json.load(f)

In [18]:
mpcnum = 1
obs = np.array(data['1'])
print(obs.shape)
print(obs[:10])

(6263, 9)
[['243.33128045114415' '-12.93348140171074' '2.260736488440913'
  '2.686224782800231' '0.13862004236294304' '-0.016276493806734113'
  '2411433.917859954' 'i00768:0' '7.246961477225209']
 ['230.25215407460462' '-15.673388431403394' '2.193428991479598'
  '2.792016817916787' '0.04677875744770939' '-0.07696526919956036'
  '2411565.614578704' 'i01460:0' '7.265208046946477']
 ['339.79264463016915' '-19.64969621754664' '2.453585749902211'
  '2.975734787065839' '0.09716686021338454' '-0.046647648899818726'
  '2411898.912928241' 'b06327:0' '7.646977313385175']
 ['340.1329971082987' '-19.848922180683267' '2.406388567194247'
  '2.97649894591633' '0.07732664488749773' '-0.05556681320607902'
  '2411902.8112824075' 'b06363:0' '7.605357443312128']
 ['340.4033650008997' '-20.09299656107993' '2.358822491042062'
  '2.977253424563898' '0.05591378618426228' '-0.06483182989951466'
  '2411906.865460648' 'b06387:0' '7.562555317693819']
 ['340.4038681362478' '-20.09358028992508' '2.3587187512438414'

In [19]:
data=[]

In [20]:
# import os

# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# fit_results = []
# preds = []
# covs = []

# outdir = "1_pngs"
# os.makedirs(outdir, exist_ok=True)

# for i in range(len(obs)): # loop over the images, plot and fit them!
#     ra, dec, rearth, rsun, dradt, ddecdt, jd, plate_sol_id, vmag = obs[i]
#     plate_id, sol_id = plate_sol_id.split(":")

#     preds.append([ra, dec, vmag])
#     fits_path = f"./{mpcnum}/{mpcnum}_{plate_id}.fits"

#     with fits.open(fits_path) as hdul:
#         hdu = next(h for h in hdul if h.data is not None)
#         raw_image = hdu.data
#         wcs = WCS(hdu.header)

#     ny_raw, nx_raw = raw_image.shape
#     N = 3

#     y0, y1 = ny_raw // N, (N - 1) * ny_raw // N
#     x0, x1 = nx_raw // N, (N - 1) * nx_raw // N
#     image = raw_image[y0:y1, x0:x1]

#     pixscale_arcsec = np.mean(proj_plane_pixel_scales(wcs)) * 3600.0
#     sigma_arcsec = 5.0
#     sigma_pix = sigma_arcsec / pixscale_arcsec

#     x_pred_raw, y_pred_raw = wcs.world_to_pixel_values(float(ra), float(dec))
#     x_pred = x_pred_raw - x0
#     y_pred = y_pred_raw - y0

#     ny, nx = image.shape

#     peak_flux = image[int(y_pred), int(x_pred)] - np.median(image)
#     length0 = sigma_pix
#     flux0 = peak_flux * length0 * np.sqrt(2 * np.pi) * sigma_pix

#     print(flux0)

#     p0 = [np.max([flux0, 0]), x_pred, y_pred, 2.0, np.deg2rad(2.0), sigma_pix, np.median(image)]

#     print(p0)

#     result, best_fit, model, cov = fit_trail(image, p0)
#     resid = image - model

#     fit_ra, fit_dec = wcs.pixel_to_world_values(best_fit["x0"] + x0, best_fit["y0"] + y0)

#     fig, axes = plt.subplots(1, 3, figsize=(24, 8), subplot_kw={"projection": wcs})

#     vmin = min(np.min(image), np.min(model))
#     vmax = max(np.max(image), np.max(model))
#     rmax = np.max(np.abs(resid))

#     im0 = axes[0].imshow(image, origin="lower", vmin=vmin, vmax=vmax)
#     axes[0].set_title("Data")
#     fig.colorbar(im0, ax=axes[0])

#     im1 = axes[1].imshow(model, origin="lower", vmin=vmin, vmax=vmax)
#     axes[1].set_title("Fit")
#     fig.colorbar(im1, ax=axes[1])

#     im2 = axes[2].imshow(resid, origin="lower", vmin=-rmax, vmax=rmax)
#     axes[2].set_title("Residual")
#     fig.colorbar(im2, ax=axes[2])

#     for ax in axes:
#         ax.scatter(x_pred, y_pred, marker="x", c="red", s=90, linewidths=2, transform=ax.get_transform("pixel"), alpha=0.1)
#         ax.set_xlabel("RA")
#         ax.set_ylabel("Dec")

#     plt.tight_layout()

#     png_path = os.path.join(outdir, f"{mpcnum}_{plate_idimport os

# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# fit_results = []
# preds = []
# covs = []

# outdir = "1_pngs"
# os.makedirs(outdir, exist_ok=True)

# for i in range(len(obs)): # loop over the images, plot and fit them!
#     ra, dec, rearth, rsun, dradt, ddecdt, jd, plate_sol_id, vmag = obs[i]
#     plate_id, sol_id = plate_sol_id.split(":")

#     preds.append([ra, dec, vmag])
#     fits_path = f"./{mpcnum}/{mpcnum}_{plate_id}.fits"

#     with fits.open(fits_path) as hdul:
#         hdu = next(h for h in hdul if h.data is not None)
#         raw_image = hdu.data
#         wcs = WCS(hdu.header)

#     ny_raw, nx_raw = raw_image.shape
#     N = 3

#     y0, y1 = ny_raw // N, (N - 1) * ny_raw // N
#     x0, x1 = nx_raw // N, (N - 1) * nx_raw // N
#     image = raw_image[y0:y1, x0:x1]

#     pixscale_arcsec = np.mean(proj_plane_pixel_scales(wcs)) * 3600.0
#     sigma_arcsec = 5.0
#     sigma_pix = sigma_arcsec / pixscale_arcsec

#     x_pred_raw, y_pred_raw = wcs.world_to_pixel_values(float(ra), float(dec))
#     x_pred = x_pred_raw - x0
#     y_pred = y_pred_raw - y0

#     ny, nx = image.shape

#     peak_flux = image[int(y_pred), int(x_pred)] - np.median(image)
#     length0 = sigma_pix
#     flux0 = peak_flux * length0 * np.sqrt(2 * np.pi) * sigma_pix

#     print(flux0)

#     p0 = [np.max([flux0, 0]), x_pred, y_pred, 2.0, np.deg2rad(2.0), sigma_pix, np.median(image)]

#     print(p0)

#     result, best_fit, model, cov = fit_trail(image, p0)
#     resid = image - model

#     fit_ra, fit_dec = wcs.pixel_to_world_values(best_fit["x0"] + x0, best_fit["y0"] + y0)

#     fig, axes = plt.subplots(1, 3, figsize=(24, 8), subplot_kw={"projection": wcs})

#     vmin = min(np.min(image), np.min(model))
#     vmax = max(np.max(image), np.max(model))
#     rmax = np.max(np.abs(resid))

#     im0 = axes[0].imshow(image, origin="lower", vmin=vmin, vmax=vmax)
#     axes[0].set_title("Data")
#     fig.colorbar(im0, ax=axes[0])

#     im1 = axes[1].imshow(model, origin="lower", vmin=vmin, vmax=vmax)
#     axes[1].set_title("Fit")
#     fig.colorbar(im1, ax=axes[1])

#     im2 = axes[2].imshow(resid, origin="lower", vmin=-rmax, vmax=rmax)
#     axes[2].set_title("Residual")
#     fig.colorbar(im2, ax=axes[2])

#     for ax in axes:
#         ax.scatter(x_pred, y_pred, marker="x", c="red", s=90, linewidths=2, transform=ax.get_transform("pixel"), alpha=0.1)
#         ax.set_xlabel("RA")
#         ax.set_ylabel("Dec")

#     plt.tight_layout()

#     png_path = os.path.join(outdir, f"{mpcnum}_{plate_id}_fit.png")
#     fig.savefig(png_path, dpi=150, bbox_inches="tight")
#     plt.close(fig)

#     fit_results.append([result, best_fit, model, cov, [fit_ra, fit_dec, best_fit["flux"]]])
#     covs.append(cov)

# }_fit.png")
#     fig.savefig(png_path, dpi=150, bbox_inches="tight")
#     plt.close(fig)

#     fit_results.append([result, best_fit, model, cov, [fit_ra, fit_dec, best_fit["flux"]]])
#     covs.append(cov)



In [22]:
import os
import gc

fit_results = []
preds = []
covs = []
pixscales_arcsec = []

outdir = "1_results"
os.makedirs(outdir, exist_ok=True)

for i in range(len(obs)): # loop over the images and fit them
    print(i)
    ra, dec, rearth, rsun, dradt, ddecdt, jd, plate_sol_id, vmag = obs[i]
    plate_id, sol_id = plate_sol_id.split(":")

    preds.append([ra, dec, vmag])
    fits_path = f"./{mpcnum}/{mpcnum}_{plate_id}.fits"

    with fits.open(fits_path, memmap=False) as hdul:
        hdu = next(h for h in hdul if h.data is not None)
        raw_image = np.array(hdu.data, copy=True)
        header = hdu.header.copy()

    wcs = WCS(header)

    ny_raw, nx_raw = raw_image.shape
    N = 3

    y0, y1 = ny_raw // N, (N - 1) * ny_raw // N
    x0, x1 = nx_raw // N, (N - 1) * nx_raw // N
    image = raw_image[y0:y1, x0:x1].copy()

    pixscale_arcsec = np.mean(proj_plane_pixel_scales(wcs)) * 3600.0
    pixscales_arcsec.append(pixscale_arcsec)

    sigma_arcsec = 5.0
    sigma_pix = sigma_arcsec / pixscale_arcsec

    x_pred_raw, y_pred_raw = wcs.world_to_pixel_values(float(ra), float(dec))
    x_pred = x_pred_raw - x0
    y_pred = y_pred_raw - y0

    ny, nx = image.shape

    peak_flux = image[int(y_pred), int(x_pred)] - np.median(image)
    length0 = sigma_pix
    flux0 = peak_flux * length0 * np.sqrt(2 * np.pi) * sigma_pix

    p0 = [np.max([flux0, 0]), x_pred, y_pred, 2.0, np.deg2rad(2.0), sigma_pix, np.median(image)]

    result, best_fit, model, cov = fit_trail(image, p0)

    fit_ra, fit_dec = wcs.pixel_to_world_values(best_fit["x0"] + x0, best_fit["y0"] + y0)

    fit_results.append([result, best_fit, cov, [fit_ra, fit_dec, best_fit["flux"]]])
    covs.append(cov)

    del raw_image, image, model, wcs, header, hdu
    gc.collect()

fits = np.array([f[-1] for f in fit_results]).astype(float)
preds = np.array(preds).astype(float)
covs = np.array(covs).astype(float)
pixscales_arcsec = np.array(pixscales_arcsec).astype(float)

np.save(os.path.join(outdir, "fits.npy"), fits)
np.save(os.path.join(outdir, "preds.npy"), preds)
np.save(os.path.join(outdir, "covs.npy"), covs)
np.save(os.path.join(outdir, "pixscales_arcsec.npy"), pixscales_arcsec)

np.savez(os.path.join(outdir, "fit_outputs.npz"), fits=fits, preds=preds, covs=covs, pixscales_arcsec=pixscales_arcsec)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

IndexError: index 353 is out of bounds for axis 1 with size 278

In [23]:
fits = np.array([f[-1] for f in fit_results]).astype(float)
preds = np.array(preds).astype(float)
covs = np.array(covs).astype(float)

np.save(os.path.join(outdir, "fits.npy"), fits)
np.save(os.path.join(outdir, "preds.npy"), preds)
np.save(os.path.join(outdir, "covs.npy"), covs)
np.savez(os.path.join(outdir, "fit_outputs.npz"), fits=fits, preds=preds, covs=covs)

In [24]:
outdir = "1_pngs"
data = np.load(os.path.join(outdir, "fit_outputs.npz"))

fits = data["fits"]
preds = data["preds"]
covs = data["covs"]

In [25]:
fit_ra, fit_dec, fit_flux = fits.T
pred_ra, pred_dec, pred_mag = preds[:-1].T

dra = (fit_ra - pred_ra) * 3600.0 * np.cos(np.deg2rad(pred_dec))  # arcsec
ddec = (fit_dec - pred_dec) * 3600.0                              # arcsec

t = np.arange(len(dra))

In [26]:
fit_ra-pred_ra

array([-1.01876913e-04, -5.79713717e-04, -1.15942620e-03,  1.09148758e-03,
        1.08461175e-03,  8.22194558e-04, -9.12182967e-04, -2.45272422e-04,
       -1.53038141e-04, -2.86123036e-04, -2.03455589e-03, -3.22604957e-04,
       -4.47141840e-04, -8.26154608e-05,  5.03240769e-02,  1.96448307e-04,
        2.75744148e-04, -1.09385103e-03,  5.15286134e-04, -6.29480223e-02,
       -2.59627052e-04,  1.09167866e-03, -4.95165971e-04,  7.71497724e-04,
        9.79395439e-04,  1.37204949e-03, -9.76194661e-04, -1.87879856e-04,
        7.68415448e-04, -4.83822104e-02, -3.59380122e-02,  2.51405730e-03,
        2.17140061e-02,  1.61222189e-04, -5.85879652e-04, -1.98248170e-04,
        6.50822763e-04, -2.14206116e-04, -2.06818682e-04, -8.76021940e-04,
       -4.31919723e-02,  1.81804844e-03, -1.78543780e-03, -1.49349369e-03,
       -8.15777854e-04,  1.16237642e-04, -1.40150032e-04, -2.05167246e-04,
        5.87856561e-04,  1.53883982e-03,  1.97443135e-03, -1.10538692e-03,
       -1.40835357e-02,  

In [27]:
fit_dec-pred_dec

array([-8.40565272e-04, -8.42472400e-04,  4.58038378e-03,  1.73144435e-04,
       -1.13454957e-03, -7.62159928e-04, -1.39492546e-04, -9.90152929e-04,
       -6.97001762e-04, -6.59235316e-04, -7.10598932e-04, -9.50130641e-04,
       -2.74872618e-04, -5.36515854e-04, -4.85819008e-02, -5.03474521e-04,
       -2.04489086e-04, -1.43466572e-04, -4.76633453e-04, -1.06849402e-03,
        1.44778890e-04, -1.84165710e-03,  5.82503506e-04, -1.20516749e-04,
       -3.15284433e-04, -5.98400221e-05, -1.86398196e-04,  2.86550612e-04,
       -2.91225557e-04,  3.75130031e-02,  3.00805541e-02,  1.27897630e-04,
       -3.12724020e-02, -8.54252439e-04, -8.51225602e-04, -1.14631918e-03,
       -5.32717164e-04, -3.52247985e-04, -2.26007784e-04, -3.74813877e-04,
       -2.95462845e-03, -6.24430706e-04, -1.44273468e-03,  6.80440195e-04,
        1.59121951e-04, -3.10698217e-04, -2.03660875e-04, -3.59844860e-04,
       -5.92406570e-04, -3.46759985e-04, -1.32434372e-04, -1.25766373e-03,
        2.96325866e-03,  

In [41]:
from scipy.stats import norm

xerr_arcsec = np.sqrt(covs[:, 1, 1]) * pixscale_arcsec
yerr_arcsec = np.sqrt(covs[:, 2, 2]) * pixscale_arcsec

uncertainty_cut_arcsec = 5.0
good = np.isfinite(dra) & np.isfinite(ddec) & np.isfinite(xerr_arcsec) & np.isfinite(yerr_arcsec) & (xerr_arcsec < uncertainty_cut_arcsec) & (yerr_arcsec < uncertainty_cut_arcsec)

t_good = t[good]
dra_good = dra[good]
ddec_good = ddec[good]
xerr_good = xerr_arcsec[good]
yerr_good = yerr_arcsec[good]

fig, axes = plt.subplots(1, 3, figsize=(56, 16))

ax = axes[0]
ax.errorbar(t_good, dra_good, yerr=xerr_good, fmt="none", capsize=2, elinewidth=1, alpha=0.6, color="tab:blue", label="dRA")
ax.errorbar(t_good, ddec_good, yerr=yerr_good, fmt="none", capsize=2, elinewidth=1, alpha=0.6, color="tab:orange", label="dDec")
ax.axhline(0, color="k", lw=1, alpha=0.3)
ax.set_ylim(-10, 10)
ax.set_xlabel("Observation index")
ax.set_ylabel("Offset [arcsec]")
ax.legend()

clip = 5.0

dra_med = np.median(dra_good)
dra_sig = 1.4826 * np.median(np.abs(dra_good - dra_med))
dra_clip = dra_good[np.abs(dra_good - dra_med) < clip * dra_sig]

ddec_med = np.median(ddec_good)
ddec_sig = 1.4826 * np.median(np.abs(ddec_good - ddec_med))
ddec_clip = ddec_good[np.abs(ddec_good - ddec_med) < clip * ddec_sig]

dra_mu, dra_std = norm.fit(dra_clip)
ddec_mu, ddec_std = norm.fit(ddec_clip)

axes[1].hist(dra_clip, bins=50, density=True, alpha=0.6, color="tab:blue")
x = np.linspace(np.min(dra_clip), np.max(dra_clip), 200)
axes[1].plot(x, norm.pdf(x, dra_mu, dra_std), color="tab:blue")
axes[1].axvline(0, color="k", lw=1, alpha=0.3)
axes[1].axvline(dra_mu, color="k", lw=1, ls="--", alpha=0.6)
axes[1].set_xlabel("dRA [arcsec]")
axes[1].set_ylabel("Density")
axes[1].set_title(f"dRA: μ={dra_mu:.2f}, σ={dra_std:.2f}")

axes[2].hist(ddec_clip, bins=50, density=True, alpha=0.6, color="tab:orange")
x = np.linspace(np.min(ddec_clip), np.max(ddec_clip), 200)
axes[2].plot(x, norm.pdf(x, ddec_mu, ddec_std), color="tab:orange")
axes[2].axvline(0, color="k", lw=1, alpha=0.3)
axes[2].axvline(ddec_mu, color="k", lw=1, ls="--", alpha=0.6)
axes[2].set_xlabel("dDec [arcsec]")
axes[2].set_ylabel("Density")
axes[2].set_title(f"dDec: μ={ddec_mu:.2f}, σ={ddec_std:.2f}")

plt.tight_layout()

png_path = os.path.join(outdir, "offset_diagnostics_uncertainty_lt_5arcsec_errorbars_only.png")
fig.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Saved {png_path}")
print(f"Kept {np.sum(good)} / {len(good)} points with both uncertainties < {uncertainty_cut_arcsec:.1f} arcsec")
print(f"Rejected {len(good) - np.sum(good)} / {len(good)} points due to large/non-finite uncertainties")
print(f"dRA  mean = {dra_mu:.3f} arcsec")
print(f"dRA  std  = {dra_std:.3f} arcsec")
print(f"dDec mean = {ddec_mu:.3f} arcsec")
print(f"dDec std  = {ddec_std:.3f} arcsec")
print(f"dRA  sigma-clipped {len(dra_good) - len(dra_clip)} / {len(dra_good)} quality-filtered points")
print(f"dDec sigma-clipped {len(ddec_good) - len(ddec_clip)} / {len(ddec_good)} quality-filtered points")

/tmp/ipykernel_73033/23413801.py:4: RuntimeWarning: invalid value encountered in sqrt
  yerr_arcsec = np.sqrt(covs[:, 2, 2]) * pixscale_arcsec


Saved 1_pngs/offset_diagnostics_uncertainty_lt_5arcsec_errorbars_only.png
Kept 738 / 795 points with both uncertainties < 5.0 arcsec
Rejected 57 / 795 points due to large/non-finite uncertainties
dRA  mean = -0.938 arcsec
dRA  std  = 7.205 arcsec
dDec mean = -0.230 arcsec
dDec std  = 7.201 arcsec
dRA  sigma-clipped 87 / 738 quality-filtered points
dDec sigma-clipped 86 / 738 quality-filtered points


In [29]:
fig, ax = plt.subplots(figsize=(6, 4))

flux_err = np.sqrt(covs[:, 0, 0])

ax.errorbar(pred_mag, fit_flux, yerr=flux_err, fmt="o", capsize=3)
ax.set_yscale("log")

ax.set_xlabel("Predicted magnitude")
ax.set_ylabel("Fitted flux")
ax.set_ylim(ymin=1e-1)

plt.tight_layout()

png_path = os.path.join(outdir, "pred_mag_vs_fit_flux.png")
fig.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Saved {png_path}")

Saved 1_pngs/pred_mag_vs_fit_flux.png
